# 第一课：走进AI工程化 —— 从基础模型到AI应用

## 学习目标
- 理解什么是AI工程化，它与传统机器学习工程的区别
- 认识基础模型（Foundation Model）的概念与发展历程
- 通过 API 调用亲身体验大语言模型的能力
- 对比不同模型的回答风格与能力差异

> 本 Notebook 基于 Chip Huyen《AI Engineering: Building Applications with Foundation Models》编写。
> 所有代码单元都配有详细中文注释，适合零编程经验的学员。

## 环境准备

> 请先运行 `00_Environment_Setup.ipynb` 完成环境配置（安装依赖包 + 设置 API Key），
> 然后再回到本 Notebook。

完成后，运行下面的代码加载环境变量：

In [ ]:
# 从 .env 文件加载 API Key（无需每次输入）
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== 选一个服务商：只改这一行，其他都不用动 =====
#   'openai'     云端  需要 OPENAI_API_KEY      效果最强，支持 Embedding
#   'deepseek'   云端  需要 DEEPSEEK_API_KEY    云端最便宜，无 Embedding
#   'openrouter' 云端  需要 OPENROUTER_API_KEY  可调用多家模型，无 Embedding
#   'ollama'     本地  不需要 Key，免费离线     先跑 `ollama serve` 并 pull 模型
PROVIDER = 'openai'

# 下面四家都兼容 OpenAI 的接口格式，区别只在：地址、Key、模型名。
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = 用 OpenAI 官方默认地址
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # 小模型：便宜、快
        'model_big': 'gpt-5.6-terra',                # 大模型：贵、强
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # 快、便宜
        'model_big': 'deepseek-v4-pro',              # 更强、更慢；V4 两个模型都会先思考再回答
        'embedding_model': None,                     # DeepSeek 目前不提供 Embedding 接口
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter 不转发 Embedding 接口
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # 本地模型不校验 Key
        'model': 'gemma4:e2b-mlx',                   # 需先 ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # 需先 ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# 先检查 Key：Key 为空时 OpenAI 客户端会直接抛出一长串报错，不容易看懂。
if not cfg['api_key']:
    raise SystemExit(
        f"没读到 '{PROVIDER}' 的 API Key。请在 .env 文件里补上 {PROVIDER.upper()}_API_KEY，\n"
        f"或者把上面的 PROVIDER 改成 'ollama'，用本地模型运行，完全不需要 Key。"
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# 后面所有代码都只用这三个变量，换服务商不需要改任何一行业务代码
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'连接成功！服务商 = {PROVIDER}，默认模型 = {MODEL}')


---

## 活动一：第一次与 AI "对话"（用代码）

### 活动目标
用 Python 代码调用 GPT 模型，让它帮你完成一个简单任务。感受"用代码控制 AI"的过程。

就像你用 ChatGPT 网页版聊天一样，只不过这次是用代码来发送消息。

In [ ]:
# 活动一：让 AI 帮你写一封邮件

# 定义你要发送给 AI 的消息
# "system" 是给 AI 的角色设定，"user" 是你提出的问题
response = client.chat.completions.create(
    model=MODEL,  # 使用 gpt-5.6-luna，性价比高，速度快
    messages=[
        {"role": "system", "content": "你是一个专业的商务助手，擅长写正式邮件。"},
        {"role": "user", "content": "帮我写一封邮件给客户，告知他们我们的新产品发布会将于下周五下午3点举行，邀请他们参加。"}
    ],
    temperature=0.7  # 控制创造力：0=保守，1=有创意
)

# 提取 AI 的回答
ai_reply = response.choices[0].message.content
print("=" * 50)
print("AI 写的邮件：")
print("=" * 50)
print(ai_reply)

### 💬 讨论

看看 AI 写的邮件，思考以下问题：
- 邮件的格式是否专业？语气是否合适？
- 有没有遗漏什么重要信息？
- 如果让你修改，你会改哪些地方？

> **小提示**：你可以修改上面代码中 `"content"` 里的文字，让 AI 帮你写不同的内容。试试看！

---

## 活动二：对比不同"角色"设定下的 AI 回答

### 活动目标
同一个问题，给 AI 不同的"角色"设定，观察回答风格的变化。理解 `system` 消息（角色设定）的力量。

In [ ]:
# 活动二：同一个问题，三种角色

question = "什么是人工智能？"

# 角色一：大学教授
print("=" * 50)
print("【角色一：大学教授】")
print("=" * 50)
response1 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "你是一位计算机科学教授，用严谨、学术的方式回答问题。"},
        {"role": "user", "content": question}
    ],
    temperature=0.5
)
print(response1.choices[0].message.content)

print()

# 角色二：小学老师
print("=" * 50)
print("【角色二：小学老师】")
print("=" * 50)
response2 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "你是一位小学老师，用简单易懂、有趣的方式向10岁的孩子解释概念。"},
        {"role": "user", "content": question}
    ],
    temperature=0.7
)
print(response2.choices[0].message.content)

print()

# 角色三：脱口秀演员
print("=" * 50)
print("【角色三：脱口秀演员】")
print("=" * 50)
response3 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "你是一位脱口秀演员，用幽默风趣的方式回答问题，带点俏皮话。"},
        {"role": "user", "content": question}
    ],
    temperature=1.2  # 更高的温度让回答更有创意
)
print(response3.choices[0].message.content)

### 💬 讨论

- 三种角色的回答有什么不同？哪种风格最适合学习？
- 同样的角色设定技巧可以用在 ChatGPT 网页版中吗？
- 你能想出其他有趣的"角色"吗？在上面的代码中试试！

---

## 活动三：探索 AI 的能力边界

### 活动目标
故意问 AI 一些"难题"，观察它的局限。理解 AI 能做什么、不能做什么。

In [ ]:
# 活动三：测试 AI 的能力边界

# 测试一：问一个非常新的时事（AI 的训练数据有截止日期）
print("【测试一：时效性问题】")
print("问题：2025年6月中国最新发布的AI政策是什么？")
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "2025年6月中国最新发布的AI政策是什么？请详细说明。"}
    ]
)
print(response.choices[0].message.content)
print()

# 测试二：问一个需要精确计算的问题
print("【测试二：数学计算】")
print("问题：12345 × 67890 = ?")
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "请心算 12345 × 67890，不要用代码，直接给出答案。"}
    ]
)
print(response.choices[0].message.content)
print("💡 实际答案：838,102,050 —— AI 可能算对也可能算错！")
print()

# 测试三：让 AI 承认"不知道"
print("【测试三：让 AI 说「我不知道」】")
print("问题：请详细描述一个名叫「张伟明」的虚构人物的完整生平。")
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "请详细描述一个名叫「张伟明」的虚构人物的完整生平。如果这个人物是编造的，请直接说不知道。"}
    ]
)
print(response.choices[0].message.content)

### 💬 讨论

- AI 在哪个测试中表现得最好？哪个最差？
- 时效性问题：AI 知道最新的事情吗？为什么？
- 数学问题：AI 是真正的"计算"还是"猜测"？
- **关键认知**：AI 是一个"文本生成器"，不是"知识库"。它的目标是生成看起来合理的文本，而不是保证事实正确。

---

## 本节回顾

| 技能 | 说明 |
|------|------|
| ✅ 调用 OpenAI API | 用代码向 GPT 发送消息并获取回复 |
| ✅ 角色设定 | 通过 `system` 消息控制 AI 的回答风格 |
| ✅ 测试 AI 边界 | 了解 AI 在时效性、数学方面的局限 |
| ✅ 模型对比 | 初步感受不同参数对输出的影响 |

### 课后练习
1. 修改活动一的邮件内容，让 AI 写不同场景的邮件（求职信、感谢信等）
2. 创造你自己的"角色"，尝试角色扮演对话
3. 搜索 "ChatGPT 有趣的角色设定提示词"，收集 3 个你觉得好玩的 prompt